# Classification Walkthrough: Bank Marketing

The deepest tour of `dscompanion` in this notebook set. We'll use the UCI **Bank
Marketing** dataset — 45,211 records from a Portuguese bank's phone marketing campaign
for term deposits — to walk through EDA, feature processing, splitting strategy,
multi-algorithm comparison (the leaderboard), hyperparameter tuning, explainability,
and exporting a governance-ready model card.

Why this dataset suits classification well: the target `y` (did the client subscribe?)
is heavily imbalanced (most calls don't convert), and the features are a realistic mix
of numeric (`age`, `balance`, `duration`) and categorical (`job`, `marital`, `education`)
columns — exactly the kind of tabular data dscompanion is built for.

See [`README.md`](README.md) for the full notebook index.

In [1]:
!pip install -q dscompanion[notebooks]

zsh:1: no matches found: dscompanion[notebooks]


## Fetch the dataset

Fetched once, then cached locally as parquet — `notebooks/.data_cache/` is gitignored,
so this never gets committed.

One thing worth doing up front: dscompanion's classification models expect a
**numeric** target (`0`/`1`), not a categorical string label — the raw `y` column here
is `"yes"`/`"no"`, so we map it before saving.

In [2]:
from pathlib import Path
import pandas as pd
from ucimlrepo import fetch_ucirepo

cache_dir = Path(".data_cache")
cache_dir.mkdir(exist_ok=True)
data_path = cache_dir / "bank_marketing.parquet"

if not data_path.exists():
    ds = fetch_ucirepo(id=222)
    df = ds.data.features.copy()
    df["y"] = ds.data.targets["y"].map({"yes": 1, "no": 0})
    df.to_parquet(data_path, index=False)
else:
    df = pd.read_parquet(data_path)

print(f"{df.shape[0]} rows, {df.shape[1]} columns")
print("Target balance:")
print(df["y"].value_counts(normalize=True))

45211 rows, 17 columns
Target balance:
y
0    0.883015
1    0.116985
Name: proportion, dtype: float64


## EDA and reporting

`PipelineConfig`'s EDA stage runs automatically as part of `PipelineRunner.run()` — no
separate call needed. Setting `reporting.html_report=True` writes a full interactive EDA
report (univariate distributions, bivariate relationships against the target, correlation
structure) to disk alongside the run's other artifacts.

## Splitting strategy

`SplitConfig(method="stratified", ...)` keeps the train/val/test target distribution
consistent — important here since `y` is imbalanced enough that a plain random split
could leave one split with meaningfully fewer positive examples than another.

In [3]:
from dscompanion.pipeline import PipelineConfig, PipelineRunner

cfg = PipelineConfig(
    name="bank_marketing_classification",
    data={"path": str(data_path), "format": "parquet", "target": "y"},
    split={"method": "stratified", "test_size": 0.2, "val_size": 0.1},
    model={"task": "classification", "algorithm": "xgboost"},
    reporting={"output_dir": "reports", "html_report": True},
)
result = PipelineRunner(cfg).run()
result.metrics

2026-09-13 11:19:54  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:54  INFO      dscompanion.pipeline.runner  PipelineRunner  |  bank_marketing_classification  v1.0


2026-09-13 11:19:54  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: classification  |  Algorithm: xgboost


2026-09-13 11:19:54  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:54  WARNING   dscompanion.pipeline.runner  Non-default config choices (will appear in report):


...

2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  Pipeline complete — 4.3s  |  Run: 20260913_111954


2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  Report: reports/20260913_111954/reports/bank_marketing_classification_v1.0_model_card.html


2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  ============================================================


,split,metric,value
0,train,f1,0.619107
1,train,precision,0.756061
2,train,recall,0.524160
3,train,roc_auc,0.950745
4,train,gini,0.901489
5,train,ks_statistic,0.766160
6,train,log_loss,0.178049
7,test,f1,0.501399
8,test,precision,0.614540
9,test,recall,0.423440


`result.metrics` is a tidy (long-format) table: one row per `(split, metric)` pair. This
shape is shared by every task dscompanion supports — only the metric names differ.
Pivoting makes it easier to compare train vs. test at a glance:

In [4]:
result.metrics.pivot(index="metric", columns="split", values="value")

split,test,train,val
metric,,,
f1,0.501399,0.619107,0.498592
gini,0.832773,0.901489,0.834841
ks_statistic,0.704600,0.766160,0.699113
log_loss,0.218092,0.178049,0.215935
precision,0.614540,0.756061,0.616725
psi,0.001125,NaN,0.001947
recall,0.423440,0.524160,0.418440
roc_auc,0.916387,0.950745,0.917421


In [5]:
# The HTML EDA report and other run artifacts live under result.run_dir
print(result.run_dir)
list(result.run_dir.glob("**/*.html"))[:5]

reports/20260913_111954


[PosixPath('reports/20260913_111954/reports/bank_marketing_classification_v1.0_model_card.html')]

## Comparing algorithms: the leaderboard

Rather than committing to `xgboost` up front, `LeaderboardConfig(enabled=True)` trains
and compares every algorithm dscompanion supports for this task, ranks them, and
promotes the top performer — `config.model.algorithm` is updated in place to match, so
everything downstream (tuning, explainability, the model card) uses the winner.

In [6]:
cfg_lb = PipelineConfig(
    name="bank_marketing_leaderboard",
    data={"path": str(data_path), "format": "parquet", "target": "y"},
    split={"method": "stratified", "test_size": 0.2, "val_size": 0.1},
    model={"task": "classification", "algorithm": "xgboost"},
    leaderboard={"enabled": True},
    reporting={"output_dir": "reports", "html_report": False},
)
result_lb = PipelineRunner(cfg_lb).run()

print("Winning algorithm:", cfg_lb.model.algorithm)
result_lb.leaderboard.leaderboard_

2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  PipelineRunner  |  bank_marketing_leaderboard  v1.0


2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: classification  |  Algorithm: xgboost


2026-09-13 11:19:58  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:58  WARNING   dscompanion.pipeline.runner  Non-default config choices (will appear in report):


...

2026-09-13 11:21:33  INFO      dscompanion.pipeline.runner  Pipeline complete — 94.9s  |  Run: 20260913_111958


2026-09-13 11:21:33  INFO      dscompanion.pipeline.runner  Report: n/a


2026-09-13 11:21:33  INFO      dscompanion.pipeline.runner  ============================================================


Winning algorithm: xgboost


,algorithm,status,fit_time_seconds,error,f1,precision,recall,roc_auc,gini,ks_statistic,log_loss,psi
0,xgboost,ok,0.835,None,0.498592,0.616725,0.418440,0.917421,0.834841,0.699113,0.215935,0.001947
1,lightgbm,ok,1.187,None,0.531714,0.619497,0.465721,0.917003,0.834006,0.698060,0.215405,0.001136
2,gradient_boosting,ok,6.123,None,0.461095,0.590406,0.378251,0.904792,0.809584,0.673358,0.229533,0.002888
3,extra_trees,ok,0.221,None,0.000000,0.000000,0.000000,0.903143,0.806286,0.670495,0.262951,0.002906
4,random_forest,ok,0.565,None,0.405616,0.596330,0.307329,0.902099,0.804198,0.670980,0.232051,0.003127
5,adaboost,ok,1.168,None,0.004695,0.333333,0.002364,0.833609,0.667218,0.523034,0.359184,0.002237
6,decision_tree,ok,0.053,None,0.409160,0.577586,0.316785,0.829123,0.658246,0.533506,0.272666,0.000229
7,naive_bayes,ok,0.008,None,0.352132,0.421053,0.302600,0.818968,0.637937,0.526496,0.562362,0.002094
8,knn,ok,0.045,None,0.256579,0.421622,0.184397,0.709758,0.419515,0.369905,1.739567,0.000450
9,svm,ok,37.731,None,0.018433,0.363636,0.009456,0.690774,0.381547,0.359898,0.362710,0.000701


## Hyperparameter tuning

`TuningConfig(enabled=True)` runs an Optuna search over the chosen algorithm's
hyperparameter space. `n_trials=20` here keeps this notebook fast to run end-to-end —
production runs would typically use 100-200 trials (see the `TuningConfig` docstring).

In [7]:
cfg_tuned = PipelineConfig(
    name="bank_marketing_tuned",
    data={"path": str(data_path), "format": "parquet", "target": "y"},
    split={"method": "stratified", "test_size": 0.2, "val_size": 0.1},
    model={"task": "classification", "algorithm": "xgboost"},
    tuning={"enabled": True, "n_trials": 20},
    reporting={"output_dir": "reports", "html_report": False},
)
result_tuned = PipelineRunner(cfg_tuned).run()

print("Best params found:")
print(result_tuned.tuner.best_params_)
result_tuned.metrics.pivot(index="metric", columns="split", values="value")

2026-09-13 11:21:33  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:21:33  INFO      dscompanion.pipeline.runner  PipelineRunner  |  bank_marketing_tuned  v1.0


2026-09-13 11:21:33  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: classification  |  Algorithm: xgboost


2026-09-13 11:21:33  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:21:33  WARNING   dscompanion.pipeline.runner  Non-default config choices (will appear in report):


...

2026-09-13 11:21:55  INFO      dscompanion.pipeline.runner  Pipeline complete — 22.3s  |  Run: 20260913_112133


2026-09-13 11:21:55  INFO      dscompanion.pipeline.runner  Report: n/a


2026-09-13 11:21:55  INFO      dscompanion.pipeline.runner  ============================================================


Best params found:
{'n_estimators': 721, 'max_depth': 8, 'learning_rate': 0.024795803650758556, 'subsample': 0.7650970966716663, 'colsample_bytree': 0.856148643942211, 'reg_alpha': 0.005351416165734628, 'reg_lambda': 0.022055966549515183, 'min_child_weight': 5}


split,test,train,val
metric,,,
f1,0.494672,0.668010,0.501401
gini,0.834184,0.918997,0.838329
ks_statistic,0.704050,0.785403,0.707970
log_loss,0.217478,0.167501,0.214757
precision,0.608276,0.815053,0.615120
psi,0.001300,NaN,0.000925
recall,0.416824,0.565914,0.423168
roc_auc,0.917092,0.959498,0.919164


## Explainability: SHAP vs. permutation importance

dscompanion supports two explainability approaches with different cost/speed tradeoffs:

- **`permutation_enabled`** (cheap, model-agnostic): shuffles each feature and measures
  the drop in performance. Fast, works for any estimator. This is the sensible default.
- **`shap_enabled`** (more expensive, more detailed): computes per-prediction feature
  attributions via SHAP values. Off by default — it's opt-in because it's genuinely
  more expensive to compute, not because anything is wrong with it.

We'll use permutation importance here since it's the lighter-weight default; SHAP is
available the same way if you want the extra detail (just flip `shap_enabled=True`).

In [8]:
cfg_explain = PipelineConfig(
    name="bank_marketing_explain",
    data={"path": str(data_path), "format": "parquet", "target": "y"},
    split={"method": "stratified", "test_size": 0.2, "val_size": 0.1},
    model={"task": "classification", "algorithm": "xgboost"},
    explain={"permutation_enabled": True},
    reporting={"output_dir": "reports", "html_report": False},
)
result_explain = PipelineRunner(cfg_explain).run()

display(result_explain.permutation_importance.importance_table())
fig = result_explain.permutation_importance.summary_plot()
fig.show()

2026-09-13 11:21:55  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:21:55  INFO      dscompanion.pipeline.runner  PipelineRunner  |  bank_marketing_explain  v1.0


2026-09-13 11:21:55  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: classification  |  Algorithm: xgboost


2026-09-13 11:21:55  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:21:55  WARNING   dscompanion.pipeline.runner  Non-default config choices (will appear in report):


...

2026-09-13 11:21:59  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:21:59  INFO      dscompanion.pipeline.runner  Pipeline complete — 3.6s  |  Run: 20260913_112155


2026-09-13 11:21:59  INFO      dscompanion.pipeline.runner  Report: n/a


2026-09-13 11:21:59  INFO      dscompanion.pipeline.runner  ============================================================


,feature,importance_mean,importance_std,rank
0,duration,0.243881,0.010439,1
1,month,0.071695,0.004838,2
2,day_of_week,0.027405,0.001799,3
3,previous,0.026014,0.002249,4
4,age,0.006459,0.000791,5
5,balance,0.003847,0.001014,6
6,campaign,0.003788,0.000854,7
7,education,0.001947,0.000565,8
8,marital,0.001690,0.000472,9
9,job,0.000745,0.000438,10


## Governance-ready model card

`result.model_card` is generated as part of every run. Exporting it produces a
self-contained report — dataset summary, EDA highlights, model configuration,
evaluation metrics, and explainability outputs — meant to be handed to a reviewer,
auditor, or teammate who wasn't in this notebook.

In [9]:
from pathlib import Path

Path("reports").mkdir(exist_ok=True)
result_explain.model_card.to_excel("reports/bank_marketing_model_card.xlsx")
result_explain.model_card.to_html("reports/bank_marketing_model_card.html")
print("Model card written to reports/")

2026-09-13 11:21:59  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 7 numeric, 9 categorical, 0 datetime, 0 boolean


2026-09-13 11:21:59  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 7 numeric, 9 categorical, 0 datetime, 0 boolean


2026-09-13 11:21:59  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 7 numeric, 9 categorical, 0 datetime, 0 boolean


2026-09-13 11:21:59  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 7 numeric, 9 categorical, 0 datetime, 0 boolean


2026-09-13 11:22:00  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 7 numeric, 9 categorical, 0 datetime, 0 boolean


2026-09-13 11:22:00  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 7 numeric, 9 categorical, 0 datetime, 0 boolean


2026-09-13 11:22:01  INFO      dscompanion.docs.model_card  ModelCard (xlsx) → reports/bank_marketing_model_card.xlsx


2026-09-13 11:22:01  INFO      dscompanion.docs.model_card  ModelCard (html) → reports/bank_marketing_model_card.html


Model card written to reports/


## Next

[`03_regression_wine_quality.ipynb`](03_regression_wine_quality.ipynb) covers the same
ground for a regression task — and one thing that's classification-only (the
leaderboard) that regression can't use yet.